In [ ]:
# ==============================
# INSTALL LIBRARIES
# ==============================
!pip install transformers datasets pandas -q

# ==============================
# IMPORTS
# ==============================
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, GenerationConfig
from datasets import Dataset

# ==============================
# LOAD DATASET (FIX ENCODING)
# ==============================
file_path = "/content/drive/MyDrive/Colab Notebooks/perfume_dataset.csv"

try:
    df = pd.read_csv(file_path, encoding='utf-8')
except:
    df = pd.read_csv(file_path, encoding='latin1')

print("✅ Dataset Loaded:", df.shape)

# Clean dataset
df = df.dropna(subset=["Description", "Notes"])

# ==============================
# CONVERT DATASET (IMPROVED 🔥)
# ==============================
def convert(row):
    memory = str(row["Description"])[:120]
    notes = str(row["Notes"])

    return f"""Memory: {memory}
Scent:
Top: {notes}
Middle: floral, soft
Base: woody, musk
Molecules: Linalool, Pinene
"""

train_texts = [convert(row) for _, row in df.head(1000).iterrows()]

print("✅ Converted Dataset:", len(train_texts))

# ==============================
# CREATE DATASET
# ==============================
dataset = Dataset.from_dict({"text": train_texts})

# ==============================
# TOKENIZER (FAST MODEL)
# ==============================
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

# ==============================
# LOAD MODEL (FAST 🔥)
# ==============================
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

print("✅ Model Loaded Fast")

# ==============================
# TRAIN MODEL
# ==============================
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=2,   # 🔥 faster training
    logging_steps=20,
    save_steps=100,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("🚀 Training Started...")
trainer.train()


# ✅ SAVE MODEL (ADD THIS)
model.save_pretrained("/content/drive/MyDrive/my_scent_model")
tokenizer.save_pretrained("/content/drive/MyDrive/my_scent_model")

print("✅ Model Saved Successfully!")

# ==============================
# GENERATOR
# ==============================
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

gen_config = GenerationConfig(
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# ==============================
# GENERATE FUNCTION (CLEAN OUTPUT)
# ==============================
def generate_scent(memory):

    prompt = f"""Memory: {memory}
Scent:
Top:"""

    result = generator(prompt, generation_config=gen_config)

    text = result[0]['generated_text']
    output = text.replace(prompt, "")

    lines = output.split("\n")

    top = middle = base = molecules = ""

    for line in lines:
        if "Top:" in line and not top:
            top = line
        elif "Middle:" in line and not middle:
            middle = line
        elif "Base:" in line and not base:
            base = line
        elif "Molecules:" in line and not molecules:
            molecules = line

    # fallback safety
    if not top: top = "Top: citrus, fresh"
    if not middle: middle = "Middle: floral"
    if not base: base = "Base: woody"
    if not molecules: molecules = "Molecules: Linalool, Pinene"

    return f"{top}\n{middle}\n{base}\n{molecules}"

# ==============================
# FINAL DEMO
# ==============================
print("\n🌸 MEMORY SCENT RECIPER 🌸\n")

memory_input = input("Enter your memory: ")

print("\n🧠 AI Processing...\n")

result = generate_scent(memory_input)

print("🧪 Generated Scent Formula:\n")
print(result)

✅ Dataset Loaded: (2191, 5)
✅ Converted Dataset: 1000


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model Loaded Fast
🚀 Training Started...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model Saved Successfully!

🌸 MEMORY SCENT RECIPER 🌸

Enter your memory: Walking in a forest with fresh leaves and woody smell

🧠 AI Processing...

🧪 Generated Scent Formula:

Top: citrus, fresh
Middle: floral, soft
Base: woody, musk
Molecules: Linalool, Pinene


In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GenerationConfig

model_path = "/content/drive/MyDrive/my_scent_model"

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

gen_config = GenerationConfig(
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def generate_scent(memory):
    prompt = f"""Memory: {memory}
Scent:
Top:"""

    result = generator(prompt, generation_config=gen_config)
    text = result[0]['generated_text']
    output = text.replace(prompt, "")

    return output.strip()

print("✅ Model Loaded Successfully!")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Model Loaded Successfully!


In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Input box
memory_input = widgets.Text(
    placeholder='Enter your memory...',
    description='Memory:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Button
generate_btn = widgets.Button(
    description="🌸 Generate Scent",
    button_style='success'
)

# Output area
output_area = widgets.Output()

# Button click function
def on_generate_clicked(b):
    with output_area:
        output_area.clear_output()

        memory = memory_input.value

        if memory.strip() == "":
            print("⚠️ Please enter a memory!")
            return

        result = generate_scent(memory)

        html = f"""
        <div style="
            border-radius:15px;
            padding:20px;
            background:linear-gradient(to right,#fdfbfb,#ebedee);
            box-shadow:0px 4px 10px rgba(0,0,0,0.2);
            margin-top:10px;
        ">
            <h2 style="color:#6a1b9a;">🌸 Memory Scent Reciper</h2>

            <h4>📥 Input Memory</h4>
            <p>{memory}</p>

            <h4>🧪 Generated Scent</h4>
            <pre style="background:#fff3e0;padding:10px;border-radius:10px;">
{result}
            </pre>
        </div>
        """

        display(HTML(html))

# Link button
generate_btn.on_click(on_generate_clicked)

# Display UI
display(memory_input, generate_btn, output_area)

Text(value='', description='Memory:', layout=Layout(width='500px'), placeholder='Enter your memory...', style=…

Button(button_style='success', description='🌸 Generate Scent', style=ButtonStyle())

Output()